# Physics-Informed Tumor Growth Forecaster

A learned reaction-diffusion model. A 3D CNN reads the source scan plus
treatment context and predicts patient-specific PDE parameters. A
differentiable Fisher-KPP simulator then evolves the true source density
forward to the target time.

    du/dt = D * laplacian(u) + rho * u * (1 - u) - kappa * u

- The physics is the forecaster; the network learns its parameters
- Trained on 208 training transitions, validated on 39 unseen transitions
- Prediction starts from the true source scan, so it departs from the
  persistence baseline only through learned dynamics

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repo and install
!rm -rf /content/BINNs
!git clone https://github.com/tanushappapogu-max/BINNs.git /content/BINNs
%cd /content/BINNs
!pip install -e '.[ml,imaging]' -q

In [ ]:
# Verify GPU availability
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Configure paths
from pathlib import Path

DRIVE_DATA = Path('/content/drive/MyDrive/STS_Project_Data/data')
DERIVED = DRIVE_DATA / 'derived' / 'mu_glioma_post'

TRAIN_INDEX = DERIVED / 'shared_training_transitions.json'
VAL_INDEX = DERIVED / 'shared_model_selection_transitions.json'
MANIFEST = DERIVED / 'shared_split_manifest.json'
DATA_ROOT = Path('/content/drive/MyDrive/STS_Project_Data')
OUTPUT_ROOT = DRIVE_DATA / 'outputs' / 'unet_pinn'

for p in [TRAIN_INDEX, VAL_INDEX, MANIFEST]:
    assert p.exists(), f'Missing: {p}'
print('All paths verified')

In [ ]:
from gbm_pinn.unet_forecaster import TrainConfig, train

# High-resolution run: 2mm voxels instead of 4mm. Dice is a boundary-sensitive
# metric, so the coarse grid used for local development caps how well any model
# can score. This is the reason to use a GPU.
#
# Defaults encode results established locally on the full cohort:
#   time_scaled=False      scan intervals are clinically driven and
#                          anti-correlated with change (r=-0.285)
#   use_cavity_domain=False cavity collapses; freezing it blocks real predictions
config = TrainConfig(
    transition_index_path=TRAIN_INDEX,
    manifest_path=MANIFEST,
    val_transition_index_path=VAL_INDEX,
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    device='cuda',
    downsample=2,
    target_shape=(120, 120, 80),
    base_filters=16,
    n_steps=30,
    dt=0.2,
    epochs=25,
    batch_size=2,
    learning_rate=3e-4,
    param_reg_weight=2.0,
    time_scaled=False,
    use_cavity_domain=False,
)

results = train(config)

In [ ]:
# Results summary
import json
r = json.load(open(OUTPUT_ROOT / 'results.json'))
print(f"Mean Dice          {r['mean_dice']:.4f}")
print(f"Mean skill         {r['mean_skill']:+.4f}")
print(f"Beating baseline   {r['n_beating_persistence']}/{r['n_total']}")
print()
print(f"GROWTH-REGION Dice {r['mean_growth_dice']:.4f}   (baseline scores 0.0000 here)")
print(f"Patients beating   {r['n_patients_beating']}/{r['n_patients']}")
if 'wilcoxon_p_patient_clustered' in r:
    print(f"Wilcoxon p         {r['wilcoxon_p_patient_clustered']:.4f}  (clustered by patient)")
print()
print("Stratified by how much the tumor actually moved:")
for g in ('high_change', 'low_change'):
    if f'{g}_n' in r:
        print(f"  {g:12s} n={r[f'{g}_n']:>3}  skill={r[f'{g}_mean_skill']:+.4f}  "
              f"beating={r[f'{g}_beating']}  growth_dice={r[f'{g}_mean_growth_dice']:.4f}")
print()
print("Volume (how much will it grow):")
print(f"  model MAE        {r.get('volume_mae_ml', 0):.2f} mL")
print(f"  baseline MAE     {r.get('persistence_volume_mae_ml', 0):.2f} mL")
print(f"  correlation      {r.get('volume_change_correlation', float('nan')):+.3f}")